# 04 — Compositions: CLIP-guided Diffusion · Transformer+CLIP+Diffusion · GAN vs Diffusion

This notebook demonstrates multi-model composition pipelines.

## Composition overview

```
  CLIP-guided Diffusion
  text prompt → CLIP text encoder → guidance signal
                                        ↓
  x_T (noise) ──[UNet + CLIP grad]──▶ x_0 (image)

  Transformer + CLIP + Diffusion
  LM generates k prompt candidates
      ↓ CLIP ranks vs desired class embedding
  best prompt → CFG diffusion → image

  GAN vs Diffusion comparison
  Same MNIST data, same config → train both → compare samples + diversity
  Diversity metric: mean per-image pixel variance
```

In [ ]:
# ── Setup (works in Colab and in a local checkout) ─────────────────────────
# Colab: clone the repo into /content and pip-install it into the kernel's
# Python (honors the repo's torch/numpy pins). Locally: walk up to the repo
# root. Either way the cwd becomes the repo root so `main.py`, `data/` and
# `runs/` resolve, and `src/` + this folder are put on sys.path so imports
# work without restarting the kernel after the editable install.
import os, pathlib, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_JUPYTER_TOKEN'))
REPO_URL = 'https://github.com/juan-garassino/mini-networks.git'
if IN_COLAB:
    root = pathlib.Path('/content/mini-networks')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)], check=True)
else:
    root = pathlib.Path.cwd().resolve()
    while not (root / 'main.py').exists() and root != root.parent:
        root = root.parent
os.chdir(root)
for p in (root / 'src', root / 'colab' / 'notebooks'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('repo root:', root)

In [ ]:
from pathlib import Path

from viz import plot_metrics, show_image_grid
from mini_networks.colab.launcher import run_composition, run_model

---
## 1. CLIP-guided Diffusion

CLIP maps a text prompt to the nearest digit class; a class-conditional UNet
then samples that class with classifier-free guidance
(`eps = (1 + w) * eps_cond - w * eps_uncond`).

In [ ]:
cgd = run_composition('clip_guided_diffusion', fast_demo=True)
print(cgd['run_dir'], 'class for "digit zero":', cgd['class_id'])

In [ ]:
plot_metrics(Path(cgd['run_dir']) / 'metrics.jsonl', keys=['clip_loss', 'diffusion_loss'])

In [ ]:
from mini_networks.compositions.clip_guided_diffusion import CLIPGuidedDiffusionConfig, CLIPGuidedDiffusion
from mini_networks.core.logging.logger import Logger
import tempfile

config = CLIPGuidedDiffusionConfig(fast_demo=True)
comp   = CLIPGuidedDiffusion()

with tempfile.TemporaryDirectory() as tmp:
    comp.train_all(config, Logger(tmp, 'cgd_demo'))   # CLIP, then the class-conditional UNet

# text -> nearest class (CLIP) -> classifier-free-guided samples of that class
class_id = comp.text_to_class('digit zero', config)
samples = comp.sample(class_id, 8, config)
show_image_grid(samples, title=f"CLIP-guided diffusion: 'digit zero' -> class {class_id}", nrow=4)

---
## 2. Transformer + CLIP + Diffusion

Three-stage pipeline:
1. Train TransformerLM on Shakespeare
2. Train CLIP on MNIST image-text pairs
3. Train conditioned UNet  

At inference: LM generates prompt candidates → CLIP ranks them → Diffusion generates image.

In [ ]:
tcd = run_composition('transformer_clip_diffusion', fast_demo=True)
print(tcd['run_dir'], 'class:', tcd['class_id'])

In [ ]:
plot_metrics(Path(tcd['run_dir']) / 'metrics.jsonl', keys=['lm_loss', 'clip_loss', 'diff_loss'])

In [ ]:
from mini_networks.compositions.transformer_clip_diffusion import (
    TransformerCLIPDiffusionConfig, TransformerCLIPDiffusion
)
from mini_networks.core.logging.logger import Logger
import tempfile

config = TransformerCLIPDiffusionConfig(fast_demo=True)
comp   = TransformerCLIPDiffusion()

with tempfile.TemporaryDirectory() as tmp:
    comp.train_all(config, Logger(tmp, 'tcd_demo'))   # LM, CLIP, conditional UNet

# Seed text -> LM prompt candidates -> CLIP ranks classes -> CFG diffusion image
images, class_id, prompts = comp.generate_image('digit zero', config)
print('LM prompts:', prompts)
show_image_grid(images, title=f'TransformerCLIPDiffusion -> class {class_id}', nrow=4)

# Rank a hand-written prompt list directly
class_id, scores = comp.rank_prompts_by_class(['digit zero', 'digit nine', 'the number one'], config)
print(f'Best class: {class_id}  scores: {[f"{s:.3f}" for s in scores]}')

---
## 3. GAN vs Diffusion comparison

Side-by-side educational comparison:
- GAN: fast inference, unstable training
- DDPM: stable MSE loss, slower inference (T steps)

**Diversity metric:** mean per-image pixel variance (higher = more varied samples)

In [ ]:
from mini_networks.compositions.gan_diffusion_comparison import GANDiffusionConfig, GANDiffusionComparison
from mini_networks.core.logging.logger import Logger
import tempfile

config = GANDiffusionConfig(fast_demo=True)
comp   = GANDiffusionComparison()

with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'gd_demo')
    comp.train_gan(config, logger)
    comp.train_diffusion(config, logger)
    plot_metrics(logger, keys=['g_loss', 'd_loss', 'diff_loss'])

In [ ]:
result = comp.compare(config, n_samples=8)

print(f"{'Metric':<25} {'GAN':>10} {'Diffusion':>12}")
print('-' * 50)
print(f"{'Diversity (pixel var)':<25} {result['gan_diversity']:>10.4f} {result['diffusion_diversity']:>12.4f}")

In [ ]:
import torch

gan_samples  = result['gan_samples']
diff_samples = result['diffusion_samples']

# Side-by-side: GAN on left, Diffusion on right
combined = torch.cat([gan_samples[:4], diff_samples[:4]], dim=0)
show_image_grid(combined, title='GAN (top row) vs Diffusion (bottom row)', nrow=4)

---
## 4. Checkpoint roundtrip

Shows that `load_checkpoint()` restores a model trained through the runtime
contract (here: DDPM) into a fresh trainer.

In [ ]:
from mini_networks.models.diffusion.config import DiffusionConfig
from mini_networks.models.diffusion.trainer import DDPMTrainer

# Train a standalone DDPM through the shared runtime contract, then reload its
# checkpoint into a FRESH trainer and sample from it.
diff_logger = run_model('diffusion', fast_demo=True)

d_config = DiffusionConfig(fast_demo=True)
d_trainer = DDPMTrainer()
d_trainer.load_checkpoint(d_config, diff_logger.artifacts_dir)
d_result = d_trainer.infer(d_config, {'n_samples': 4})
print(f'Diffusion checkpoint: samples shape = {d_result["samples"].shape}')